# 1. Veri Setinin İndirilmesi ve Hazırlanması


In [ ]:
!pip install opencv-python-headless numpy scipy scikit-image matplotlib pandas

In [ ]:
import os
import zipfile
import requests

url = "https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip"

response = requests.get(url)

with open("Fluo-N2DH-GOWT1.zip", "wb") as file:
    file.write(response.content)

with zipfile.ZipFile("Fluo-N2DH-GOWT1.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset downloaded and extracted successfully!")

In [ ]:
for root, dirs, files in os.walk("dataset"):
    print(root, "->", len(files), "files")

# 2. Görüntü Analizi ve Histogram


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

image_dir = "dataset/Fluo-N2DH-GOWT1/01"

image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.endswith(".tif")
])

image_path = os.path.join(image_dir, image_files[0])
image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

plt.figure(figsize=(8, 6))
plt.imshow(image, cmap="gray")
plt.title("First Microscopic Image")
plt.axis("off")
plt.show()

print("Image name:", image_files[0])
print("Image shape:", image.shape)
print("Data type:", image.dtype)
print("Minimum intensity:", image.min())
print("Maximum intensity:", image.max())

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(image.ravel(), bins=256, range=(0, 256))

plt.title("Image Intensity Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")

plt.show()

In [ ]:
import numpy as np

print("Mean intensity:", np.mean(image))
print("Median intensity:", np.median(image))
print("Standard deviation:", np.std(image))
print("25th percentile:", np.percentile(image, 25))
print("75th percentile:", np.percentile(image, 75))

# 3. Görüntü Karelerinin Seçimi ve Gerçek Hücre Sayıları


In [ ]:
student_id_last5 = 12902
seed = student_id_last5 + 2026

print("Random Seed:", seed)

In [ ]:
import os

base_dir = "dataset/Fluo-N2DH-GOWT1"

for sequence in ["01", "02"]:
    folder = os.path.join(base_dir, sequence)

    files = sorted([
        f for f in os.listdir(folder)
        if f.endswith(".tif")
    ])

    print(f"Sequence {sequence}")
    print("Number of frames:", len(files))
    print("First 5 frames:", files[:5])
    print("Last 5 frames:", files[-5:])
    print("----------------------")

In [ ]:
import numpy as np
import pandas as pd

student_id_last5 = 12902
seed = student_id_last5 + 2026

rng = np.random.default_rng(seed)

# Development and Validation from Sequence 01
all_indices = np.arange(92)

# Randomly select 10 Development frames
# and 5 Validation frames with at least 5 unused frames between groups
for attempt in range(10000):
    development = np.sort(rng.choice(all_indices, 10, replace=False))

    validation_candidates = np.array([
        i for i in all_indices
        if np.all(np.abs(development - i) >= 6)
    ])

    if len(validation_candidates) >= 5:
        validation = np.sort(
            rng.choice(validation_candidates, 5, replace=False)
        )
        break
else:
    raise RuntimeError("Could not select valid frames.")

# Select 3 non-overlapping Test blocks,
# each containing 5 consecutive frames,
# with at least 5 unused frames between blocks.
for attempt in range(10000):
    possible_starts = np.arange(88)
    starts = np.sort(rng.choice(possible_starts, 3, replace=False))

    if np.all(np.diff(starts) >= 10):
        break
else:
    raise RuntimeError("Could not select valid Test blocks.")

test_frames = []

for block_number, start in enumerate(starts, start=1):
    for i in range(start, start + 5):
        test_frames.append((i, block_number))

# Create CSV records
records = []

for i in development:
    records.append({
        "filename": f"t{i:03d}.tif",
        "sequence": "01",
        "frame_index": int(i),
        "group": "Development",
        "test_block": "",
        "seed": seed
    })

for i in validation:
    records.append({
        "filename": f"t{i:03d}.tif",
        "sequence": "01",
        "frame_index": int(i),
        "group": "Validation",
        "test_block": "",
        "seed": seed
    })

for i, block_number in test_frames:
    records.append({
        "filename": f"t{i:03d}.tif",
        "sequence": "02",
        "frame_index": int(i),
        "group": "Test",
        "test_block": block_number,
        "seed": seed
    })

selected_df = pd.DataFrame(records)

selected_df.to_csv("selected_frames.csv", index=False)

print("Development:", development)
print("Validation:", validation)
print("Test block starts:", starts)
print("\nSelected frames:")
print(selected_df.to_string(index=False))

print("\nTotal selected frames:", len(selected_df))
print("CSV saved successfully!")

In [ ]:
print("Development count:", len(development))
print("Validation count:", len(validation))
print("Test count:", len(test_frames))

print(
    "Minimum distance between Development and Validation:",
    min(abs(d - v) for d in development for v in validation)
)

print(
    "Test block gaps:",
    np.diff(starts) - 5
)

In [ ]:
import os
import cv2

gt_dir = "dataset/Fluo-N2DH-GOWT1/01_GT/TRA"

gt_files = sorted([
    f for f in os.listdir(gt_dir)
    if f.endswith(".tif")
])

print("Number of GT files:", len(gt_files))
print("First 10 files:", gt_files[:10])

gt_path = os.path.join(gt_dir, gt_files[0])
gt_image = cv2.imread(gt_path, cv2.IMREAD_UNCHANGED)

print("GT image name:", gt_files[0])
print("GT image shape:", gt_image.shape)
print("GT data type:", gt_image.dtype)
print("Unique labels:", len(set(gt_image.ravel())))

In [ ]:
import numpy as np

positive_labels = np.unique(gt_image)
positive_labels = positive_labels[positive_labels > 0]

print("Background label: 0")
print("Number of unique cell labels:", len(positive_labels))
print("Cell IDs:", positive_labels)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

gt_records = []

development_validation_df = selected_df[
    selected_df["group"].isin(["Development", "Validation"])
]

for _, row in development_validation_df.iterrows():

    sequence = row["sequence"]
    frame_index = row["frame_index"]

    gt_dir = f"dataset/Fluo-N2DH-GOWT1/{sequence}_GT/TRA"
    gt_filename = f"man_track{frame_index:03d}.tif"
    gt_path = os.path.join(gt_dir, gt_filename)

    gt = cv2.imread(gt_path, cv2.IMREAD_UNCHANGED)

    if gt is None:
        raise FileNotFoundError(gt_path)

    labels = np.unique(gt)
    labels = labels[labels > 0]

    gt_records.append({
        "filename": row["filename"],
        "sequence": sequence,
        "frame_index": frame_index,
        "group": row["group"],
        "test_block": row["test_block"],
        "ground_truth_count": len(labels)
    })

gt_df = pd.DataFrame(gt_records)

gt_df.to_csv("ground_truth_counts.csv", index=False)

print(gt_df.to_string(index=False))
print("\nTotal frames:", len(gt_df))
print("Ground Truth counts saved successfully!")

# 4. Temel Yöntem ile Hücre Sayımı




In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Load one Development image
image_path = "dataset/Fluo-N2DH-GOWT1/01/t012.tif"
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Apply Otsu thresholding
threshold_value, binary = cv2.threshold(
    image,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Count connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    binary,
    connectivity=8
)

# Exclude background
predicted_count = num_labels - 1

print("Image: t012.tif")
print("Otsu threshold:", threshold_value)
print("Predicted cell count:", predicted_count)

# Display results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(binary, cmap="gray")
plt.title("Otsu Binary Segmentation")
plt.axis("off")

plt.show()

In [ ]:
# Get component areas (excluding background)
areas = stats[1:, cv2.CC_STAT_AREA]

print("Total components:", len(areas))
print("Small components (area < 20):", np.sum(areas < 20))
print("Components with area < 50:", np.sum(areas < 50))
print("Components with area < 100:", np.sum(areas < 100))
print("Largest component area:", areas.max())
print("Smallest component area:", areas.min())
print("Average component area:", areas.mean())

In [ ]:
# Filter connected components by area
min_area = 20

filtered_binary = np.zeros_like(binary)

filtered_count = 0

for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]

    if area >= min_area:
        filtered_binary[labels == i] = 255
        filtered_count += 1

print("Minimum area:", min_area)
print("Original component count:", predicted_count)
print("Filtered component count:", filtered_count)
print("Ground Truth count:", 23)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(binary, cmap="gray")
plt.title("Original Otsu")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(filtered_binary, cmap="gray")
plt.title("After Removing Small Components")
plt.axis("off")

plt.show()

# 5. Geliştirme Aşaması Deneyleri


In [ ]:
min_areas = [10, 20, 30, 50, 100]

print("Min Area | Predicted Count | Ground Truth | Absolute Error")
print("-" * 60)

for min_area in min_areas:

    filtered_count = np.sum(areas >= min_area)

    error = abs(filtered_count - 23)

    print(
        f"{min_area:8} | "
        f"{filtered_count:15} | "
        f"{23:12} | "
        f"{error:14}"
    )

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

min_areas = [10, 20, 30, 50, 100]

dev_results = []

for _, row in gt_df[gt_df["group"] == "Development"].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    _, binary_img = cv2.threshold(
        image, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    num_labels, labels_img, stats_img, centroids = (
        cv2.connectedComponentsWithStats(
            binary_img, connectivity=8
        )
    )

    component_areas = stats_img[1:, cv2.CC_STAT_AREA]

    for min_area in min_areas:

        predicted = np.sum(component_areas >= min_area)
        actual = row["ground_truth_count"]

        dev_results.append({
            "filename": row["filename"],
            "min_area": min_area,
            "predicted_count": int(predicted),
            "ground_truth": int(actual),
            "absolute_error": int(abs(predicted - actual))
        })

dev_df = pd.DataFrame(dev_results)

print(dev_df.to_string(index=False))

print("\nAverage MAE for each Min Area:")

print(
    dev_df.groupby("min_area")["absolute_error"]
    .mean()
)

In [ ]:
dev_df.to_csv("experiment_01_area_filter.csv", index=False)

area_summary = (
    dev_df.groupby("min_area")["absolute_error"]
    .mean()
    .reset_index(name="MAE")
)

area_summary.to_csv("experiment_01_summary.csv", index=False)

print("Experiment 01 saved successfully!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

transform_results = []

def contrast_stretch(img):
    p1, p99 = np.percentile(img, (1, 99))

    if p99 <= p1:
        return img.copy()

    stretched = np.clip(
        (img.astype(np.float32) - p1) * 255 / (p99 - p1),
        0, 255
    )

    return stretched.astype(np.uint8)


clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

for _, row in gt_df[gt_df["group"] == "Development"].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    transformed_images = {
        "Original": image,
        "Contrast Stretching": contrast_stretch(image),
        "Histogram Equalization": cv2.equalizeHist(image),
        "CLAHE": clahe.apply(image)
    }

    for method, processed in transformed_images.items():

        _, binary_img = cv2.threshold(
            processed, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        num_labels, labels_img, stats_img, _ = (
            cv2.connectedComponentsWithStats(
                binary_img, connectivity=8
            )
        )

        areas = stats_img[1:, cv2.CC_STAT_AREA]

        predicted = int(np.sum(areas >= 50))
        actual = int(row["ground_truth_count"])

        transform_results.append({
            "filename": row["filename"],
            "method": method,
            "predicted_count": predicted,
            "ground_truth": actual,
            "absolute_error": abs(predicted - actual)
        })

transform_df = pd.DataFrame(transform_results)

print(transform_df.to_string(index=False))

print("\nAverage MAE by method:")

print(
    transform_df.groupby("method")["absolute_error"]
    .mean()
    .sort_values()
)

In [ ]:
# Save Experiment 01
dev_df.to_csv("experiment_01_area_filter.csv", index=False)

area_summary = (
    dev_df.groupby("min_area")["absolute_error"]
    .mean()
    .reset_index(name="MAE")
)
area_summary.to_csv("experiment_01_summary.csv", index=False)

# Save Experiment 02
transform_df.to_csv("experiment_02_intensity_transforms.csv", index=False)

transform_summary = (
    transform_df.groupby("method")["absolute_error"]
    .mean()
    .reset_index(name="MAE")
    .sort_values("MAE")
)
transform_summary.to_csv("experiment_02_summary.csv", index=False)

print("Experiments 01 and 02 saved successfully!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

filter_results = []

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

for _, row in gt_df[gt_df["group"] == "Development"].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    enhanced = clahe.apply(image)

    filtered_images = {
        "No Filter": enhanced,
        "Gaussian 3x3": cv2.GaussianBlur(enhanced, (3, 3), 0),
        "Gaussian 5x5": cv2.GaussianBlur(enhanced, (5, 5), 0),
        "Median 3x3": cv2.medianBlur(enhanced, 3),
        "Sharpening": cv2.filter2D(
            enhanced, -1, sharpen_kernel
        )
    }

    for method, processed in filtered_images.items():

        _, binary_img = cv2.threshold(
            processed,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        num_labels, labels_img, stats_img, centroids = (
            cv2.connectedComponentsWithStats(
                binary_img,
                connectivity=8
            )
        )

        areas = stats_img[1:, cv2.CC_STAT_AREA]
        predicted = int(np.sum(areas >= 50))

        actual = int(row["ground_truth_count"])

        filter_results.append({
            "filename": row["filename"],
            "filter": method,
            "predicted_count": predicted,
            "ground_truth": actual,
            "absolute_error": abs(predicted - actual)
        })

filter_df = pd.DataFrame(filter_results)

print(filter_df.to_string(index=False))

print("\nAverage MAE by filter:")

filter_summary = (
    filter_df.groupby("filter")["absolute_error"]
    .mean()
    .sort_values()
)

print(filter_summary)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

segmentation_results = []

for _, row in gt_df[gt_df["group"] == "Development"].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    methods = {}

    # Fixed Threshold
    for threshold_value in [25, 35, 45]:
        _, binary = cv2.threshold(
            image, threshold_value, 255, cv2.THRESH_BINARY
        )
        methods[f"Fixed {threshold_value}"] = binary

    # Otsu Threshold
    _, otsu_binary = cv2.threshold(
        image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    methods["Otsu"] = otsu_binary

    # Adaptive Threshold
    adaptive_binary = cv2.adaptiveThreshold(
        image,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        51,
        2
    )
    methods["Adaptive"] = adaptive_binary

    for method_name, binary in methods.items():

        num_labels, labels, stats, centroids = (
            cv2.connectedComponentsWithStats(binary, connectivity=8)
        )

        areas = stats[1:, cv2.CC_STAT_AREA]
        predicted = int(np.sum(areas >= 50))
        actual = int(row["ground_truth_count"])

        segmentation_results.append({
            "filename": row["filename"],
            "method": method_name,
            "predicted_count": predicted,
            "ground_truth": actual,
            "absolute_error": abs(predicted - actual)
        })

segmentation_df = pd.DataFrame(segmentation_results)

print(segmentation_df.to_string(index=False))

print("\nAverage MAE by segmentation method:")
print(
    segmentation_df.groupby("method")["absolute_error"]
    .mean()
    .sort_values()
)

In [ ]:
dev_df.to_csv("experiment_01_area_filter.csv", index=False)
transform_df.to_csv("experiment_02_intensity_transforms.csv", index=False)
segmentation_df.to_csv("experiment_04_segmentation.csv", index=False)

print("Previous experiments saved successfully!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

morphology_results = []

for _, row in gt_df[gt_df["group"] == "Development"].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Fixed Threshold = 25
    _, binary = cv2.threshold(
        image, 25, 255, cv2.THRESH_BINARY
    )

    for kernel_size in [3, 5]:

        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (kernel_size, kernel_size)
        )

        operations = {
            "Erosion": cv2.erode(binary, kernel),
            "Dilation": cv2.dilate(binary, kernel),
            "Opening": cv2.morphologyEx(
                binary, cv2.MORPH_OPEN, kernel
            ),
            "Closing": cv2.morphologyEx(
                binary, cv2.MORPH_CLOSE, kernel
            )
        }

        for operation_name, processed in operations.items():

            num_labels, labels, stats, centroids = (
                cv2.connectedComponentsWithStats(
                    processed, connectivity=8
                )
            )

            areas = stats[1:, cv2.CC_STAT_AREA]
            predicted = int(np.sum(areas >= 50))
            actual = int(row["ground_truth_count"])

            morphology_results.append({
                "filename": row["filename"],
                "operation": operation_name,
                "kernel_size": kernel_size,
                "predicted_count": predicted,
                "ground_truth": actual,
                "absolute_error": abs(predicted - actual)
            })

morphology_df = pd.DataFrame(morphology_results)

print(morphology_df.to_string(index=False))

print("\nAverage MAE:")
print(
    morphology_df.groupby(
        ["operation", "kernel_size"]
    )["absolute_error"]
    .mean()
    .sort_values()
)

In [ ]:
morphology_df.to_csv(
    "experiment_05_morphology.csv",
    index=False
)

print("Morphology experiment saved!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

order_results = []

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

for _, row in gt_df[gt_df["group"] == "Development"].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    _, binary = cv2.threshold(
        image, 25, 255, cv2.THRESH_BINARY
    )

    processed_images = {
        "No Morphology": binary,
        "Closing then Opening": cv2.morphologyEx(
            cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel),
            cv2.MORPH_OPEN,
            kernel
        ),
        "Opening then Closing": cv2.morphologyEx(
            cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel),
            cv2.MORPH_CLOSE,
            kernel
        )
    }

    for method, processed in processed_images.items():

        num_labels, labels, stats, centroids = (
            cv2.connectedComponentsWithStats(
                processed, connectivity=8
            )
        )

        areas = stats[1:, cv2.CC_STAT_AREA]
        predicted = int(np.sum(areas >= 50))
        actual = int(row["ground_truth_count"])

        order_results.append({
            "filename": row["filename"],
            "method": method,
            "predicted_count": predicted,
            "ground_truth": actual,
            "absolute_error": abs(predicted - actual)
        })

order_df = pd.DataFrame(order_results)

print(order_df.to_string(index=False))

print("\nAverage MAE by operation order:")

print(
    order_df.groupby("method")["absolute_error"]
    .mean()
    .sort_values()
)

In [ ]:
order_df.to_csv(
    "experiment_06_morphology_order.csv",
    index=False
)

print("Experiment 06 saved successfully!")

# 6. Görüntü İşleme Hatlarının Karşılaştırılması




In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import time

pipeline_results = []

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

# Sharpening kernel
sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
], dtype=np.float32)


def run_pipeline(image, pipeline_name):

    if pipeline_name == "Pipeline 1":
        processed = image.copy()

    elif pipeline_name == "Pipeline 2":
        processed = clahe.apply(image)

    elif pipeline_name == "Pipeline 3":
        processed = cv2.filter2D(
            image, -1, sharpen_kernel
        )

    # Fixed Threshold = 25
    _, binary = cv2.threshold(
        processed, 25, 255, cv2.THRESH_BINARY
    )

    # Opening
    opened = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )

    # Closing
    final_binary = cv2.morphologyEx(
        opened, cv2.MORPH_CLOSE, kernel
    )

    # Connected Components
    num_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            final_binary, connectivity=8
        )
    )

    areas = stats[1:, cv2.CC_STAT_AREA]

    predicted_count = int(np.sum(areas >= 50))

    return predicted_count


for _, row in gt_df[
    gt_df["group"] == "Development"
].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(
        image_path, cv2.IMREAD_GRAYSCALE
    )

    actual = int(row["ground_truth_count"])

    for pipeline_name in [
        "Pipeline 1",
        "Pipeline 2",
        "Pipeline 3"
    ]:

        start_time = time.time()

        predicted = run_pipeline(
            image, pipeline_name
        )

        processing_time = time.time() - start_time

        pipeline_results.append({
            "filename": row["filename"],
            "pipeline": pipeline_name,
            "predicted_count": predicted,
            "ground_truth": actual,
            "absolute_error": abs(
                predicted - actual
            ),
            "processing_time": processing_time
        })


pipeline_df = pd.DataFrame(pipeline_results)

print(pipeline_df.to_string(index=False))

print("\nPipeline Summary:")

summary = pipeline_df.groupby("pipeline").agg(
    MAE=("absolute_error", "mean"),
    Average_Time=("processing_time", "mean")
).reset_index()

print(summary.to_string(index=False))

In [ ]:
pipeline_df.to_csv(
    "development_pipeline_results.csv",
    index=False
)

summary.to_csv(
    "development_pipeline_summary.csv",
    index=False
)

print("Development pipeline results saved!")

# 7. Doğrulama ve Nihai İşlem Hattının Seçimi


In [ ]:
import time
import numpy as np
import pandas as pd
import os
import cv2

validation_results = []

for _, row in gt_df[
    gt_df["group"] == "Validation"
].iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(
        image_path, cv2.IMREAD_GRAYSCALE
    )

    actual = int(row["ground_truth_count"])

    for pipeline_name in [
        "Pipeline 1",
        "Pipeline 2",
        "Pipeline 3"
    ]:

        start_time = time.time()

        predicted = run_pipeline(
            image, pipeline_name
        )

        processing_time = time.time() - start_time

        validation_results.append({
            "filename": row["filename"],
            "pipeline": pipeline_name,
            "predicted_count": predicted,
            "ground_truth": actual,
            "absolute_error": abs(predicted - actual),
            "squared_error": (predicted - actual) ** 2,
            "processing_time": processing_time
        })


validation_df = pd.DataFrame(validation_results)

print("Validation Results:")
print(validation_df.to_string(index=False))

validation_summary = validation_df.groupby(
    "pipeline"
).agg(
    MAE=("absolute_error", "mean"),
    MSE=("squared_error", "mean"),
    Average_Time=("processing_time", "mean")
).reset_index()

validation_summary["RMSE"] = np.sqrt(
    validation_summary["MSE"]
)

validation_summary = validation_summary[
    ["pipeline", "MAE", "RMSE", "Average_Time"]
]

print("\nValidation Summary:")
print(validation_summary.to_string(index=False))

In [ ]:
validation_df.to_csv(
    "validation_results.csv",
    index=False
)

validation_summary.to_csv(
    "validation_summary.csv",
    index=False
)

print("Validation results saved successfully!")

# 8. Test Tahminlerinin Kilitlenmesi


In [ ]:
import json

best = validation_summary.sort_values(
    by=["MAE", "RMSE", "Average_Time"]
).iloc[0]

final_pipeline = {
    "pipeline_name": "Pipeline 1",
    "input": "Grayscale original image",
    "preprocessing": "None",
    "segmentation": {
        "method": "Fixed Threshold",
        "threshold": 25
    },
    "morphology": {
        "operation_order": ["Opening", "Closing"],
        "kernel_shape": "ELLIPSE",
        "kernel_size": [3, 3]
    },
    "connected_components": {
        "connectivity": 8,
        "minimum_area": 50
    },
    "counting": "Number of connected components with area >= 50",
    "selected_using": "Validation MAE",
    "validation_MAE": float(best["MAE"]),
"validation_RMSE": float(best["RMSE"])
}

with open("final_pipeline.json", "w") as f:
    json.dump(final_pipeline, f, indent=4)

print("Final pipeline locked and saved!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import time

test_rows = selected_df[
    selected_df["group"] == "Test"
].copy()

locked_predictions = []

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

for _, row in test_rows.iterrows():

    image_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        row["sequence"],
        row["filename"]
    )

    image = cv2.imread(
        image_path, cv2.IMREAD_GRAYSCALE
    )

    # -------------------------
    # Baseline: Otsu + CC + Area 50
    # -------------------------

    start = time.perf_counter()

    _, baseline_binary = cv2.threshold(
        image, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    _, _, baseline_stats, _ = (
        cv2.connectedComponentsWithStats(
            baseline_binary, connectivity=8
        )
    )

    baseline_areas = baseline_stats[
        1:, cv2.CC_STAT_AREA
    ]

    baseline_count = int(
        np.sum(baseline_areas >= 50)
    )

    baseline_time = time.perf_counter() - start

    # -------------------------
    # Final Pipeline 1
    # -------------------------

    start = time.perf_counter()

    _, binary = cv2.threshold(
        image, 25, 255, cv2.THRESH_BINARY
    )

    opened = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )

    closed = cv2.morphologyEx(
        opened, cv2.MORPH_CLOSE, kernel
    )

    _, _, final_stats, _ = (
        cv2.connectedComponentsWithStats(
            closed, connectivity=8
        )
    )

    final_areas = final_stats[
        1:, cv2.CC_STAT_AREA
    ]

    final_count = int(
        np.sum(final_areas >= 50)
    )

    final_time = time.perf_counter() - start

    # Save predictions only — no Ground Truth
    locked_predictions.append({
        "filename": row["filename"],
        "sequence": row["sequence"],
        "frame_index": int(row["frame_index"]),
        "test_block": row["test_block"],
        "baseline_prediction": baseline_count,
        "baseline_time": baseline_time,
        "final_prediction": final_count,
        "final_time": final_time
    })


test_predictions_df = pd.DataFrame(locked_predictions)

test_predictions_df.to_csv(
    "test_predictions_locked.csv",
    index=False
)

print("Locked test predictions saved!")
print(test_predictions_df.to_string(index=False))

In [ ]:
# Verification of Locked Test Predictions
import pandas as pd

df = pd.read_csv("test_predictions_locked.csv")

print("Number of Test Images:", len(df))
print("Missing Values:", df.isnull().sum().sum())
print(df[["filename", "baseline_prediction", "final_prediction"]])


In [ ]:

import os
import cv2
import numpy as np
import pandas as pd

predictions = pd.read_csv("test_predictions_locked.csv")

gt_records = []

for _, row in predictions.iterrows():
    frame_index = int(row["frame_index"])
    sequence = str(row["sequence"]).zfill(2)

    gt_path = os.path.join(
        "dataset/Fluo-N2DH-GOWT1",
        f"{sequence}_GT",
        "TRA",
        f"man_track{frame_index:03d}.tif"
    )

    gt = cv2.imread(gt_path, cv2.IMREAD_UNCHANGED)

    if gt is None:
        raise FileNotFoundError(gt_path)

    ground_truth_count = int(
        np.sum(np.unique(gt) > 0)
    )

    gt_records.append({
        "filename": row["filename"],
        "sequence": sequence,
        "ground_truth_count": ground_truth_count
    })

test_gt_df = pd.DataFrame(gt_records)

assert len(test_gt_df) == 15

print("Test Ground Truth Frames:", len(test_gt_df))


# 9. Bağımsız Test Değerlendirmesi


In [ ]:
import pandas as pd
import numpy as np

# Read locked predictions
predictions = pd.read_csv("test_predictions_locked.csv")

# Get Test Ground Truth counts
ground_truth = test_gt_df.copy()

# Ensure sequence values have the same format
predictions["sequence"] = (
    predictions["sequence"].astype(str).str.zfill(2)
)

ground_truth["sequence"] = (
    ground_truth["sequence"].astype(str).str.zfill(2)
)

# Match predictions with Ground Truth
test_results = predictions.merge(
    ground_truth[["filename", "sequence", "ground_truth_count"]],
    on=["filename", "sequence"],
    how="inner"
)

# Calculate errors
test_results["baseline_error"] = (
    test_results["baseline_prediction"] - test_results["ground_truth_count"]
)
test_results["final_error"] = (
    test_results["final_prediction"] - test_results["ground_truth_count"]
)

test_results["baseline_absolute_error"] = test_results["baseline_error"].abs()
test_results["final_absolute_error"] = test_results["final_error"].abs()

test_results["baseline_squared_error"] = test_results["baseline_error"] ** 2
test_results["final_squared_error"] = test_results["final_error"] ** 2

# Calculate evaluation metrics
summary_rows = []

for method in ["baseline", "final"]:
    prediction_col = f"{method}_prediction"
    error_col = f"{method}_error"
    abs_error_col = f"{method}_absolute_error"
    squared_error_col = f"{method}_squared_error"
    time_col = f"{method}_time"

    summary_rows.append({
        "method": method.capitalize(),
        "MAE": test_results[abs_error_col].mean(),
        "RMSE": np.sqrt(test_results[squared_error_col].mean()),
        "Bias": test_results[error_col].mean(),
        "Exact_count_Accuracy": (
            test_results[prediction_col] ==
            test_results["ground_truth_count"]
        ).mean() * 100,
        "Average_processing_time": test_results[time_col].mean()
    })

test_summary = pd.DataFrame(summary_rows)

# Save results
test_results.to_csv("test_results.csv", index=False)
test_summary.to_csv("test_summary.csv", index=False)

print("TEST RESULTS")
display(test_results)

print("\nTEST SUMMARY")
display(test_summary)

print("\nBoth CSV files saved successfully!")

assert len(test_results) == 15, "Expected 15 test frames"
assert test_results["ground_truth_count"].notna().all()

print("Test evaluation completed successfully!")

# 10. Zamansal Analiz ve Görselleştirmeler


In [ ]:

import matplotlib.pyplot as plt
import os

os.makedirs("test_plots", exist_ok=True)

blocks = sorted(test_results["test_block"].unique())

for block in blocks:
    block_data = test_results[
        test_results["test_block"] == block
    ].sort_values("frame_index")

    plt.figure(figsize=(9, 5))

    plt.plot(
        block_data["frame_index"],
        block_data["ground_truth_count"],
        marker="o",
        label="Ground Truth"
    )

    plt.plot(
        block_data["frame_index"],
        block_data["baseline_prediction"],
        marker="s",
        label="Baseline"
    )

    plt.plot(
        block_data["frame_index"],
        block_data["final_prediction"],
        marker="^",
        label="Final Pipeline"
    )

    plt.title(f"Test Block {block}: Cell Count Comparison")
    plt.xlabel("Frame Number")
    plt.ylabel("Cell Count")
    plt.xticks(block_data["frame_index"])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        f"test_plots/test_block_{block}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

print("All three temporal plots saved successfully!")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image = cv2.imread(
    "dataset/Fluo-N2DH-GOWT1/01/t012.tif",
    cv2.IMREAD_GRAYSCALE
)

_, threshold = cv2.threshold(
    image, 25, 255, cv2.THRESH_BINARY
)

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

opening = cv2.morphologyEx(
    threshold, cv2.MORPH_OPEN, kernel
)

closing = cv2.morphologyEx(
    opening, cv2.MORPH_CLOSE, kernel
)

num_labels, labels, stats, centroids = (
    cv2.connectedComponentsWithStats(closing, 8)
)

detected = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
count = 0

for i in range(1, num_labels):
    if stats[i, cv2.CC_STAT_AREA] >= 50:
        x, y, w, h, area = stats[i]
        cv2.rectangle(
            detected, (x, y), (x + w, y + h),
            (0, 255, 0), 2
        )
        count += 1

images = [image, threshold, opening, closing, detected]
titles = [
    "Original",
    "Fixed Threshold (25)",
    "Opening (3x3)",
    "Closing (3x3)",
    f"Final Detection: {count} Cells"
]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.savefig("final_pipeline_stages.png", dpi=300)
plt.show()

Şekil: Nihai görüntü işleme hattının aşamaları.

Orijinal mikroskobik görüntüye sabit eşikleme (25), morfolojik açma ve kapama işlemleri uygulanmıştır. Ardından bağlı bileşen analizi ve minimum alan filtresi kullanılarak hücreler tespit edilmiştir. Örnek görüntüde 23 gerçek hücreye karşılık 22 hücre tespit edilmiştir.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load a Development image
image = cv2.imread(
    "dataset/Fluo-N2DH-GOWT1/01/t012.tif",
    cv2.IMREAD_GRAYSCALE
)

# Contrast Stretching
p_low, p_high = np.percentile(image, (2, 98))

contrast = np.clip(
    (image.astype(np.float32) - p_low)
    * 255 / max(p_high - p_low, 1),
    0, 255
).astype(np.uint8)

# Histogram Equalization
hist_eq = cv2.equalizeHist(image)

# CLAHE
clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)
clahe_image = clahe.apply(image)

# Images and titles
images = [
    image,
    contrast,
    hist_eq,
    clahe_image
]

titles = [
    "Original",
    "Contrast Stretching",
    "Histogram Equalization",
    "CLAHE"
]

# Plot images and histograms
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i in range(4):
    axes[0, i].imshow(
        images[i], cmap="gray", vmin=0, vmax=255
    )
    axes[0, i].set_title(titles[i])
    axes[0, i].axis("off")

    axes[1, i].hist(
        images[i].ravel(),
        bins=256,
        range=(0, 256)
    )
    axes[1, i].set_title(titles[i] + " Histogram")
    axes[1, i].set_xlabel("Pixel Intensity")
    axes[1, i].set_ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    "intensity_histogram_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Intensity comparison figure saved!")

Şekil: Kontrast iyileştirme yöntemlerinin ve histogramlarının karşılaştırılması.

Orijinal mikroskobik görüntüye kontrast germe, histogram eşitleme ve CLAHE yöntemleri uygulanmıştır. Kontrast germe yöntemi hücrelerin görünürlüğünü artırırken, histogram eşitleme arka plan gürültüsünü de belirginleştirmiştir. CLAHE yöntemi ise yerel kontrastı daha kontrollü biçimde iyileştirmiştir. Deneysel sonuçlarda, Otsu eşikleme ve alan filtreleme ile birlikte CLAHE en düşük ortalama mutlak hatayı (MAE = 3,1) sağlamıştır.

In [ ]:
import cv2
import matplotlib.pyplot as plt

image = cv2.imread(
    "dataset/Fluo-N2DH-GOWT1/01/t012.tif",
    cv2.IMREAD_GRAYSCALE
)

# Segmentation methods
_, fixed = cv2.threshold(
    image, 25, 255, cv2.THRESH_BINARY
)

_, otsu = cv2.threshold(
    image, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

adaptive = cv2.adaptiveThreshold(
    image, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    51, 2
)

# Morphological operations
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

erosion = cv2.erode(fixed, kernel)
dilation = cv2.dilate(fixed, kernel)

opening = cv2.morphologyEx(
    fixed, cv2.MORPH_OPEN, kernel
)

closing = cv2.morphologyEx(
    fixed, cv2.MORPH_CLOSE, kernel
)

images = [
    image, fixed, otsu, adaptive,
    erosion, dilation, opening, closing
]

titles = [
    "Original",
    "Fixed Threshold (25)",
    "Otsu",
    "Adaptive Threshold",
    "Erosion (3x3)",
    "Dilation (3x3)",
    "Opening (3x3)",
    "Closing (3x3)"
]

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for ax, img, title in zip(axes.flat, images, titles):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    "segmentation_morphology_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Segmentation and morphology comparison saved!")

Şekil: Segmentasyon ve morfolojik işlemlerin karşılaştırılması.

Mikroskobik görüntü üzerinde sabit eşikleme, Otsu ve adaptif eşikleme yöntemleri karşılaştırılmıştır. Adaptif eşikleme yönteminin arka plan bölgelerini de ön plana çıkardığı gözlemlenmiştir. Ayrıca aşındırma, genişletme, açma ve kapama işlemlerinin hücre bölgeleri üzerindeki etkileri incelenmiştir. Morfolojik işlemlerin hücre boyutlarını, sınırlarını ve bağlantılarını değiştirdiği görülmüştür.

## Deney Özeti ve Proje Dosyaları


In [ ]:
variables = [
    "dev_df",
    "transform_df",
    "filter_df",
    "segmentation_df",
    "morphology_df",
    "order_df",
    "pipeline_df"
]

for name in variables:
    if name in globals():
        print(f"{name}: Available")
    else:
        print(f"{name}: Missing")

In [ ]:
import pandas as pd
import numpy as np

experiments = []

experiment_sources = [
    ("Area Filtering", "dev_df", ["min_area"]),
    ("Intensity Enhancement", "transform_df", ["method"]),
    ("Filtering", "filter_df", ["filter"]),
    ("Segmentation", "segmentation_df", ["method"]),
    ("Morphology", "morphology_df", ["operation", "kernel_size"]),
    ("Morphology Order", "order_df", ["method"]),
    ("Complete Pipeline", "pipeline_df", ["pipeline"])
]

for category, variable_name, preferred_columns in experiment_sources:

    df = globals()[variable_name].copy()

    # Detect actual parameter columns
    parameter_columns = [
        col for col in preferred_columns
        if col in df.columns
    ]

    if not parameter_columns:
        print(f"Check column names for {variable_name}:")
        print(df.columns.tolist())
        continue

    grouped = df.groupby(
        parameter_columns,
        dropna=False
    )

    for parameters, group in grouped:

        if not isinstance(parameters, tuple):
            parameters = (parameters,)

        parameter_text = ", ".join(
            f"{col}={value}"
            for col, value in zip(parameter_columns, parameters)
        )

        experiments.append({
            "category": category,
            "parameters": parameter_text,
            "frames_evaluated": len(group),
            "MAE": group["absolute_error"].mean()
        })

experiments_df = pd.DataFrame(experiments)

experiments_df.insert(
    0,
    "experiment_id",
    [
        f"EXP_{i:03d}"
        for i in range(1, len(experiments_df) + 1)
    ]
)

experiments_df.to_csv(
    "experiments.csv",
    index=False
)

print("Total experiment configurations:", len(experiments_df))

display(experiments_df)

print("experiments.csv saved successfully!")

In [ ]:
import os

required_files = [
    "selected_frames.csv",
    "experiments.csv",
    "validation_results.csv",
    "final_pipeline.json",
    "test_predictions_locked.csv",
    "test_results.csv",
    "test_summary.csv"
]

for filename in required_files:
    if os.path.exists(filename):
        print("OK     ", filename)
    else:
        print("MISSING", filename)

In [ ]:
requirements = """numpy
pandas
opencv-python-headless
scipy
scikit-image
matplotlib
requests
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully!")
print(requirements)

# 11. Sonuçlar ve Genel Değerlendirme


Nihai görüntü işleme hattı, doğrulama sonuçlarına göre seçilmiştir (MAE = 1,8). Bağımsız test verilerinde temel yöntem (Baseline) 7,27 MAE değerine ulaşırken, nihai görüntü işleme hattının MAE değeri 13,67 olmuştur.

Her iki yöntemin de gerçek hücre sayısını olduğundan düşük tahmin ettiği gözlemlenmiştir. Bu sonuçlar, doğrulama verilerinde başarılı olan nihai yöntemin Sequence 02 üzerinde aynı başarıyı gösteremediğini ve genelleme performansının sınırlı kaldığını ortaya koymaktadır.
